# Phase II validation: Agentic AI Multi-Agent Decision Support Platform for Industrial IoT Predictive Maintenance

This notebook reproduces every result in `results/report.md` and runs the platform on a live-style stream.
It works in **Google Colab** and in **VS Code** (Jupyter extension).

| Experiment | Question | Compared with |
| --- | --- | --- |
| E1 | Can one model predict 11 inverter and 8 pump-drive alarms before they trip? | ASPIRE-style Transformer, raw-feature XGBoost |
| E2 | Does the same model work on 3 kW / 10 kW and 3 HP / 10 HP units with only a new profile? | Same baselines |
| E3 | How well does the Forecast Agent estimate time to a thermal trip? | Persistence, seasonal naive |
| E4 | Does the Advisor return the right manual entry for each alarm? | README-style RAG on raw sensor values |

**Runtime:** quick mode about 2 minutes on CPU. Full mode took about 34 minutes on a 2-core CPU, almost all of it the ASPIRE-style Transformer baseline; in Colab choose *Runtime > Change runtime type > GPU* to cut that sharply. The pre-computed full results are already in `results/`.

## 1. Setup
In Colab, either set `GIT_URL` to your repository or leave it empty and upload `pdm_agentic.zip` when asked. In VS Code, open the repository folder and run the cell as is.

In [ ]:
import os, sys, pathlib, subprocess, zipfile
IN_COLAB = "google.colab" in sys.modules
GIT_URL = ""   # e.g. "https://github.com/<you>/pdm_agentic.git"

if IN_COLAB:
    if GIT_URL:
        subprocess.run(["git", "clone", GIT_URL, "/content/pdm_agentic"], check=True)
    elif not pathlib.Path("/content/pdm_agentic").exists():
        from google.colab import files
        uploaded = files.upload()                      # choose pdm_agentic.zip
        zipfile.ZipFile(next(iter(uploaded))).extractall("/content")
    ROOT = pathlib.Path("/content/pdm_agentic")
else:
    here = pathlib.Path.cwd()
    ROOT = here.parent if here.name == "notebooks" else here

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repository:", ROOT)

In [ ]:
# Install dependencies (Colab already has numpy, pandas, torch, matplotlib)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost", "pyyaml", "fastapi", "httpx"], check=True)
INSTALL_CHRONOS = False   # True to also test the Chronos-Bolt forecaster (downloads a small model)
if INSTALL_CHRONOS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "chronos-forecasting"], check=True)

## 2. Run all experiments
`QUICK = True` runs a short check; `False` reproduces the full report.

In [ ]:
QUICK = os.environ.get("PDM_QUICK", "0") == "1"   # set True for a fast check
from pdm.experiments import run_all
res = run_all(quick=QUICK)
OUT = res["out"]

## 3. E1 and E2: early warning on the trained rating and on unseen ratings

In [ ]:
import pandas as pd
from pdm.eval.report import METHOD_SHORT
pd.set_option("display.precision", 3)
s = res["summary"].copy(); s["method"] = s["method"].map(METHOD_SHORT)
s[["asset", "rating", "method", "macro_f1", "macro_f1_std", "macro_pr_auc", "event_recall", "median_lead_min", "false_alerts_per_day"]]

In [ ]:
from IPython.display import Image, display, Markdown
display(Image(filename=str(OUT / "fig_cross_rating.png")))

## 4. Per-alarm results for our method, lead time, and one worked example

In [ ]:
pa = res["per_alarm"]; trained = ["inverter_hybrid_5kw", "pump_drive_5hp"]
(pa[(pa.method == "ours_pu_xgb") & pa.rating.isin(trained)]
   .groupby(["asset", "alarm"])[["precision", "recall", "f1", "event_recall", "median_lead_min"]].mean())

In [ ]:
display(Image(filename=str(OUT / "fig_lead_time.png")))
display(Image(filename=str(OUT / "fig_example_INV-11.png")))

## 5. E3 Forecast Agent and E4 Advisor Agent

In [ ]:
display(res["forecast"].pivot_table(index=["asset", "signal"], columns="method", values="mae_pu"))
tt = res["time_to_trip"]
display(tt.groupby(["asset", "method"]).agg(caught_rate=("caught", "mean"), abs_err_min=("abs_err_min", "mean")))

In [ ]:
adv = res["advisor"]
adv.groupby("asset")[["ours_hit1", "ours_hit3", "readme_rag_hit1", "readme_rag_hit3"]].mean()

## 6. Live-style demo: the 5 kW model on a 10 kW inverter, profile swap only

In [ ]:
from pdm.api_helpers import platform_for
from pdm.twin import generate
from pdm.agents.advisor import format_checklist
plat = platform_for("inverter_hybrid_10kw", model_dir=OUT / "models")
stream = generate("inverter_hybrid_10kw", days=2, seed=2026, n_true=3, n_near=1)["raw"]
last, shown = None, 0
for t in range(60, len(stream), 5):
    out = plat.step(stream.iloc[max(0, t - 120):t])
    if out["state"] != last and out["state"] != "NORMAL":
        items = out["alarms"] or out["warnings"]
        print(out["time"], out["state"], [(i["code"], i["name"]) for i in items])
        if out["advice"] and shown < 2:
            print(format_checklist(out["advice"])); print(); shown += 1
    last = out["state"]

## 7. Optional: check on a real labelled fault dataset
Download GPVS-Faults or the inverter-driven PMSM dataset (links in `data/real/README.md`), reshape to `timestamp, run_id, label, <signals>`, then run:

In [ ]:
REAL_CSV = ""   # e.g. "data/real/pmsm_faults.csv"
if REAL_CSV:
    subprocess.run([sys.executable, "scripts/validate_real.py", "--csv", REAL_CSV], check=True)

## 8. Reading the results
- **Claim 1 (multi-fault early warning):** E1 macro-F1, event recall and lead time for 19 alarm types, against the ASPIRE-style Transformer on the same data.
- **Claim 2 (rating-agnostic):** E2 drop from trained to unseen ratings; a small drop for ours and a larger one for raw-input baselines supports the claim.
- **Claim 3 (alarm to action):** E4 hit rate of the Advisor against README-style RAG.
- **Limitations:** twin data with injected faults and placeholder thresholds; confirm on real data (section 7) and replace knowledge-base sources with manual pages.